In [ ]:
import pandas as pd
import cv2
import os
import numpy as np
from pathlib import Path
import shutil
import random
from PIL import Image
from sklearn.model_selection import train_test_split
import albumentations as A


In [9]:
PATH = "C:\\Users\\Ale\\Downloads\\DepaY-All-Files\\Codigo\\Cocina\\Kitchen-Cleanliness-Prediction-using-CNN\\final_images"

ORIGINAL_SPLIT_PATH = "No_augmented_images_split"
AUGMENTED_PATH = "Augmented_images"
AUGMENTED_SPLIT_PATH = "Augmented_images_split"


# **Original Data**

In [10]:
def split_dataset(
    input_dir,
    output_dir,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=42
):
    """
    Splits dataset into train/val/test folders with stratification.

    Expected structure:
    input_dir/
        clean/
            img1.jpg
            ...
        dirty/
            img2.jpg
            ...

    Output:
    output_dir/
        train/
            clean/
            dirty/
        val/
            clean/
            dirty/
        test/
            clean/
            dirty/
    """

    # Validate proportions
    total = train_size + val_size + test_size
    if abs(total - 1.0) > 1e-6:
        raise ValueError("train_size + val_size + test_size must equal 1")

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    classes = [d.name for d in input_dir.iterdir() if d.is_dir()]

    image_paths = []
    labels = []

    # Read all images
    for cls in classes:
        class_dir = input_dir / cls

        for img_path in class_dir.iterdir():
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                image_paths.append(img_path)
                labels.append(cls)

    # First split: train vs temp
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths,
        labels,
        test_size=(1 - train_size),
        stratify=labels,
        random_state=random_state
    )

    # Second split: val vs test
    relative_test_size = test_size / (val_size + test_size)

    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths,
        temp_labels,
        test_size=relative_test_size,
        stratify=temp_labels,
        random_state=random_state
    )

    splits = {
        "train": (train_paths, train_labels),
        "val": (val_paths, val_labels),
        "test": (test_paths, test_labels)
    }

    # Create folders and copy images
    for split_name, (paths, lbls) in splits.items():

        for path, label in zip(paths, lbls):

            target_dir = output_dir / split_name / label
            target_dir.mkdir(parents=True, exist_ok=True)

            shutil.copy2(path, target_dir / path.name)

    print("Dataset successfully split.")
    print(f"Train: {len(train_paths)}")
    print(f"Validation: {len(val_paths)}")
    print(f"Test: {len(test_paths)}")

In [11]:
split_dataset(
    input_dir=PATH,
    output_dir=ORIGINAL_SPLIT_PATH,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15
)

Dataset successfully split.
Train: 209
Validation: 45
Test: 46


# **Augmented Data**

In [15]:


def augment_existing_split(
    original_split_dir,
    output_dir,
    augmentations_per_image=3
):
    """
    Duplicates an existing split dataset and augments ONLY train.

    Original structure:
    original_split_dir/
        train/
            clean/
            dirty/

        val/
            clean/
            dirty/

        test/
            clean/
            dirty/

    Output:
    output_dir/
        train/   <-- augmented
        val/     <-- unchanged copy
        test/    <-- unchanged copy
    """

    original_split_dir = Path(original_split_dir)
    output_dir = Path(output_dir)

    # ----------------------------------------
    # REMOVE OLD OUTPUT
    # ----------------------------------------

    if output_dir.exists():
        shutil.rmtree(output_dir)

    # ----------------------------------------
    # COPY ENTIRE DATASET
    # ----------------------------------------

    shutil.copytree(
        original_split_dir,
        output_dir
    )

    print("Dataset duplicated.")

    # ----------------------------------------
    # AUGMENTATION PIPELINE
    # ----------------------------------------

    transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=30, p=0.7),
        A.RandomBrightnessContrast(p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.RandomScale(scale_limit=0.1, p=0.5),
    ])

    train_dir = output_dir / "train"

    classes = [
        d.name for d in train_dir.iterdir()
        if d.is_dir()
    ]

    total_augmented = 0

    # ----------------------------------------
    # AUGMENT TRAIN ONLY
    # ----------------------------------------

    for cls in classes:

        class_dir = train_dir / cls

        image_paths = list(class_dir.iterdir())

        for img_path in image_paths:

            if img_path.suffix.lower() not in [
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
                ".webp"
            ]:
                continue

            image = Image.open(img_path).convert("RGB")
            image_np = np.array(image)

            for i in range(augmentations_per_image):

                augmented = transform(image=image_np)

                augmented_image = augmented["image"]

                augmented_pil = Image.fromarray(
                    augmented_image
                )

                new_name = (
                    f"{img_path.stem}_aug_{i}.jpg"
                )

                augmented_pil.save(
                    class_dir / new_name
                )

                total_augmented += 1

    print("\nAugmentation completed.")
    print(f"Generated augmented images: {total_augmented}")

In [16]:
augment_existing_split(
    original_split_dir=ORIGINAL_SPLIT_PATH,
    output_dir=AUGMENTED_SPLIT_PATH,
    augmentations_per_image=5
)


Dataset duplicated.

Augmentation completed.
Generated augmented images: 1045
